# Track B — Sparse SINDy Policy via NN Distillation

Train a neural-network policy directly on the full-order MuJoCo simulator using
the 6-dim physical state, then distill it into a sparse SINDy polynomial
control law `π(x) → u`. The NN is an intermediate step; the deliverable is the
sparse polynomial.

```
ReducedObsEnv  ──►  PPO (train in real MuJoCo)  ──►  trained NN policy
                                                              │
                                               roll out, collect (x, u) pairs
                                                              │
                                          SINDy regression: u = Θ(x) @ coeff
                                                              │
                                           Evaluate sparse π(x)→u in real sim
```

**No surrogate is built.** Track A builds the SINDy dynamics surrogate;
Phase 3 repeats this distillation step inside Track A's surrogate.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import pysindy as ps
from tqdm.auto import tqdm
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback, CheckpointCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy

# ── Config ─────────────────────────────────────────────────────────────────────
ENV_ID          = "InvertedDoublePendulum-v5"
TOTAL_STEPS     = 400_000
N_ENVS          = 8
N_EVAL_ENVS     = 4
N_STEPS         = 2048
BATCH_SIZE      = 64
ENT_COEF        = 0.001
GAMMA           = 0.99
LEARNING_RATE   = 3e-4
NET_ARCH        = [64, 64]
N_EVAL_EPISODES = 20
EVAL_FREQ       = max(1, 25_000 // N_ENVS)
CHECKPOINT_DIR  = "checkpoints/trackB"
MODEL_PATH      = "checkpoints/trackB/ppo_trackB"

SINDY_DEG       = 2       # polynomial degree for SINDy feature library
SINDY_THRS      = 0.01    # STLSQ sparsity threshold
N_DISTIL_STEPS  = 100_000 # env steps for distillation dataset
N_SINDY_EVAL    = 20      # evaluation episodes for sparse policy

STATE_LABELS    = ["x", "θ₁", "θ₂", "ẋ", "θ̇₁", "θ̇₂"]

_env      = gym.make(ENV_ID)
MAX_STEPS = _env.spec.max_episode_steps
DT        = _env.unwrapped.dt
_env.close()

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"MAX_STEPS = {MAX_STEPS}  |  DT = {DT} s  |  max episode = {MAX_STEPS * DT:.1f} s")

In [ ]:
def obs_to_state6(obs):
    """9-dim MuJoCo observation → 6-dim state [x, θ₁, θ₂, ẋ, θ̇₁, θ̇₂].

    Angles recovered via arctan2 to avoid the sin/cos redundancy that inflates
    the SINDy polynomial feature count.
    """
    return np.array([
        obs[0],
        np.arctan2(obs[1], obs[3]),   # θ₁ = arctan2(sin θ₁, cos θ₁)
        np.arctan2(obs[2], obs[4]),   # θ₂ = arctan2(sin θ₂, cos θ₂)
        obs[5],
        obs[6],
        obs[7],
    ], dtype=np.float64)


def plot_results(ep_lengths, ep_rewards, eval_steps, eval_lengths, eval_rewards, total_steps):
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    (ax_len, ax_rew), (ax_ep_len, ax_ep_rew) = axes

    ax_len.axhspan(MAX_STEPS, MAX_STEPS * 1.05, color="green", alpha=0.15, zorder=0,
                   label=f"Task complete  ({MAX_STEPS} steps)")
    ax_len.plot(eval_steps, eval_lengths, color="steelblue", lw=1.5)
    ax_len.fill_between(eval_steps, eval_lengths, alpha=0.15, color="steelblue")
    ax_len.axhline(MAX_STEPS, color="green", ls="--", lw=1.5)
    breakthrough = eval_steps[eval_lengths >= MAX_STEPS]
    if len(breakthrough):
        ax_len.axvline(breakthrough[0], color="red", ls=":", lw=1,
                       label=f"first solve @ {breakthrough[0]:.2f}M steps")
    ax_len.set_ylim(0, MAX_STEPS * 1.04)
    ax_len.set_xlabel("Training steps (M)"); ax_len.set_ylabel("Mean episode length")
    ax_len.set_title("Learning curve — episode length"); ax_len.legend(fontsize=8)

    ax_rew.plot(eval_steps, eval_rewards, color="coral", lw=1.5)
    ax_rew.fill_between(eval_steps, eval_rewards, alpha=0.15, color="coral")
    ax_rew.axhline(0, color="gray", ls="--", lw=1)
    if len(breakthrough):
        ax_rew.axvline(breakthrough[0], color="red", ls=":", lw=1)
    ax_rew.set_xlabel("Training steps (M)"); ax_rew.set_ylabel("Mean cumulative reward")
    ax_rew.set_title("Learning curve — reward")

    eps = range(1, len(ep_lengths) + 1)
    ax_ep_len.axhspan(MAX_STEPS, MAX_STEPS * 1.05, color="green", alpha=0.15, zorder=0,
                      label=f"Task complete  ({MAX_STEPS} steps)")
    ax_ep_len.bar(eps, ep_lengths, color="steelblue", zorder=2)
    ax_ep_len.axhline(np.mean(ep_lengths), color="red", ls="--", zorder=3,
                      label=f"mean = {np.mean(ep_lengths):.0f}")
    ax_ep_len.axhline(MAX_STEPS, color="green", ls="--", lw=1.5, zorder=3)
    ax_ep_len.set_ylim(0, MAX_STEPS * 1.04)
    ax_ep_len.set_xlabel("Episode"); ax_ep_len.set_ylabel("Steps")
    ax_ep_len.set_title("Final eval — episode length"); ax_ep_len.legend(fontsize=8)

    ax_ep_rew.bar(eps, ep_rewards, color="coral", zorder=2)
    ax_ep_rew.axhline(np.mean(ep_rewards), color="red", ls="--", zorder=3,
                      label=f"mean = {np.mean(ep_rewards):.1f}")
    ax_ep_rew.axhline(0, color="gray", ls=":", alpha=0.7, zorder=3)
    ax_ep_rew.set_xlabel("Episode"); ax_ep_rew.set_ylabel("Cumulative reward")
    ax_ep_rew.set_title("Final eval — episode reward"); ax_ep_rew.legend(fontsize=8)

    plt.suptitle(
        f"Track B — NN policy (6-dim state)  |  {total_steps/1e6:.2f}M steps  |  "
        f"max episode = {MAX_STEPS} steps ({MAX_STEPS*DT:.0f} s)",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

---

## Part 1 — NN Policy on Full-Order MuJoCo

`ReducedObsEnv` wraps `InvertedDoublePendulum-v5` to expose the 6-dim
physical state `[x, θ₁, θ₂, ẋ, θ̇₁, θ̇₂]` instead of the 9-dim observation.
The reduction is necessary for SINDy: using raw sin/cos pairs in a polynomial
feature library doubles the feature count and introduces collinearity.
The reward and termination conditions are unchanged.

In [ ]:
from gymnasium import spaces


class ReducedObsEnv(gym.ObservationWrapper):
    """Expose 6-dim physical state instead of 9-dim MuJoCo observation."""

    def __init__(self, env):
        super().__init__(env)
        self.observation_space = spaces.Box(
            -np.inf, np.inf, shape=(6,), dtype=np.float64
        )

    def observation(self, obs):
        return obs_to_state6(obs)

### Training

PPO runs in `N_ENVS` parallel `ReducedObsEnv` instances — training is entirely
in the real MuJoCo simulator, no surrogate.  This policy is an intermediate
step; the deliverable is the sparse SINDy polynomial distilled from it.

In [ ]:
class _TqdmCallback(BaseCallback):
    """tqdm progress bar + in-memory training history."""

    def __init__(self, total_steps: int):
        super().__init__(verbose=0)
        self._pbar = tqdm(total=total_steps, unit="step", desc="PPO")
        self.history: dict = {"timesteps": [], "ep_len": [], "ep_rew": []}

    def _on_step(self) -> bool:
        return True

    def _on_rollout_end(self) -> None:
        self._pbar.n = self.num_timesteps
        if self.model.ep_info_buffer:
            ep_len = np.mean([ep["l"] for ep in self.model.ep_info_buffer])
            ep_rew = np.mean([ep["r"] for ep in self.model.ep_info_buffer])
            self._pbar.set_postfix(ep_len=f"{ep_len:.0f}", ep_rew=f"{ep_rew:.1f}", refresh=False)
            self.history["timesteps"].append(self.num_timesteps)
            self.history["ep_len"].append(float(ep_len))
            self.history["ep_rew"].append(float(ep_rew))
        self._pbar.refresh()

    def _on_training_end(self) -> None:
        self._pbar.n = self._pbar.total
        self._pbar.refresh()
        self._pbar.close()


vec_env  = make_vec_env(ENV_ID, n_envs=N_ENVS, wrapper_class=ReducedObsEnv)
eval_env = make_vec_env(ENV_ID, n_envs=N_EVAL_ENVS, wrapper_class=ReducedObsEnv)

eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=CHECKPOINT_DIR,
    log_path=CHECKPOINT_DIR,
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=True,
    verbose=0,
)

checkpoint_cb = CheckpointCallback(
    save_freq=max(1, 200_000 // N_ENVS),
    save_path=CHECKPOINT_DIR,
    name_prefix="ckpt",
    verbose=0,
)

tqdm_cb = _TqdmCallback(TOTAL_STEPS)
t_train_start = time.perf_counter()

model = PPO(
    "MlpPolicy",
    vec_env,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    ent_coef=ENT_COEF,
    gamma=GAMMA,
    learning_rate=LEARNING_RATE,
    policy_kwargs=dict(net_arch=NET_ARCH),
    verbose=0,
)

model.learn(total_timesteps=TOTAL_STEPS, callback=[eval_cb, checkpoint_cb, tqdm_cb])
model.save(MODEL_PATH)
vec_env.close()
eval_env.close()

nn_train_wall = time.perf_counter() - t_train_start
print(f"Done — {TOTAL_STEPS/1e6:.2f}M steps in {nn_train_wall/60:.1f} min  "
      f"| final ep_len {tqdm_cb.history['ep_len'][-1]:.0f} / {MAX_STEPS}")

In [ ]:
best       = os.path.join(CHECKPOINT_DIR, "best_model.zip")
model_path = best if os.path.exists(best) else MODEL_PATH + ".zip"

eval_env = make_vec_env(ENV_ID, n_envs=N_EVAL_ENVS, wrapper_class=ReducedObsEnv)
nn_model = PPO.load(model_path, env=eval_env)
print(f"Evaluating: {model_path}")

ep_rewards, ep_lengths = evaluate_policy(
    nn_model, eval_env,
    n_eval_episodes=N_EVAL_EPISODES,
    return_episode_rewards=True,
    deterministic=True,
)
eval_env.close()

print(f"Mean length  : {np.mean(ep_lengths):.1f} / {MAX_STEPS} steps ({np.mean(ep_lengths)*DT:.1f} s)")
print(f"Mean reward  : {np.mean(ep_rewards):.2f} +/- {np.std(ep_rewards):.2f}")
print(f"Perfect (={MAX_STEPS}): {100*np.mean(np.array(ep_lengths)==MAX_STEPS):.0f}%")

evals = np.load(os.path.join(CHECKPOINT_DIR, "evaluations.npz"))
plot_results(
    ep_lengths, ep_rewards,
    evals["timesteps"] / 1e6,
    evals["ep_lengths"].mean(axis=1),
    evals["results"].mean(axis=1),
    TOTAL_STEPS,
)

---

## Part 2 — Collect Distillation Dataset

Roll out the trained NN policy for `N_DISTIL_STEPS` steps to collect
`(state, action)` pairs.  The stochastic policy (`deterministic=False`) is used
to diversify state coverage — deterministic rollouts concentrate near the
steady-state trajectory and undersample the regions the SINDy polynomial
needs to cover.

In [ ]:
best       = os.path.join(CHECKPOINT_DIR, "best_model.zip")
model_path = best if os.path.exists(best) else MODEL_PATH + ".zip"
nn_model   = PPO.load(model_path)

rollout_env = ReducedObsEnv(gym.make(ENV_ID))

X_states  = []
U_actions = []
total     = 0

while total < N_DISTIL_STEPS:
    obs, _ = rollout_env.reset()
    done   = False
    while not done and total < N_DISTIL_STEPS:
        action, _ = nn_model.predict(obs, deterministic=False)
        X_states.append(obs.copy())
        U_actions.append(action.copy())
        obs, _, terminated, truncated, _ = rollout_env.step(action)
        done   = terminated or truncated
        total += 1

rollout_env.close()
X_states  = np.array(X_states)
U_actions = np.array(U_actions).reshape(-1, 1)

print(f"Dataset: {len(X_states):,} (state, action) pairs")
print(f"  X_states  : {X_states.shape}")
print(f"  U_actions : {U_actions.shape}")
print(f"  Action range: [{U_actions.min():.3f}, {U_actions.max():.3f}]")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

pairs = [
    (0, 3, "x (m)",    "ẋ (m/s)"),
    (1, 2, "θ₁ (rad)", "θ₂ (rad)"),
    (1, 4, "θ₁ (rad)", "θ̇₁ (rad/s)"),
    (2, 5, "θ₂ (rad)", "θ̇₂ (rad/s)"),
    (0, 1, "x (m)",    "θ₁ (rad)"),
    (3, 4, "ẋ (m/s)",  "θ̇₁ (rad/s)"),
]

for ax, (i, j, lx, ly) in zip(axes.flat, pairs):
    ax.scatter(X_states[:, i], X_states[:, j], s=1, alpha=0.3, rasterized=True)
    ax.set_xlabel(lx); ax.set_ylabel(ly)

fig.suptitle("Track B — distillation dataset state-space coverage (NN stochastic rollout)")
plt.tight_layout()
plt.show()

---

## Part 3 — Sparse SINDy Policy Distillation

Fit a sparse polynomial `u = Θ(x) @ coeff` via STLSQ regression on the
`(state, action)` dataset.  `Θ(x)` is the degree-2 polynomial feature library
over the 6-dim state.  STLSQ iteratively zeroes small coefficients, yielding
a sparse, interpretable control law.

In [ ]:
# Build polynomial feature matrix
library = ps.PolynomialLibrary(degree=SINDY_DEG)
library.fit(X_states)
Theta         = library.transform(X_states)
feature_names = library.get_feature_names_out(STATE_LABELS)

# Sparse regression: u = Θ @ coeff
optimizer = ps.STLSQ(threshold=SINDY_THRS)
optimizer.fit(Theta, U_actions)
coeff = optimizer.coef_          # shape: (1, n_features)

n_features = Theta.shape[1]
nonzero    = int(np.sum(np.abs(coeff) > 1e-8))
print(f"Polynomial features : {n_features}")
print(f"Nonzero terms       : {nonzero}  (sparsity: {100*(1 - nonzero/n_features):.0f}%)")

# In-sample R²
u_pred = (Theta @ coeff.T).flatten()
ss_res = np.sum((U_actions.flatten() - u_pred)**2)
ss_tot = np.sum((U_actions.flatten() - U_actions.mean())**2)
r2     = 1 - ss_res / ss_tot
rmse   = float(np.sqrt(np.mean((U_actions.flatten() - u_pred)**2)))
print(f"In-sample RMSE      : {rmse:.4f}")
print(f"In-sample R²        : {r2:.4f}")

In [ ]:
print("Sparse SINDy policy  π(x) → u:\n")
terms = [
    f"  {c:+.4f} · {name}"
    for name, c in zip(feature_names, coeff[0])
    if abs(c) > 1e-8
]
if terms:
    print("u = \n" + "\n".join(terms))
else:
    print("u = 0  (all terms zeroed — lower SINDY_THRS)")


def sindy_policy(state: np.ndarray) -> np.ndarray:
    """Evaluate the sparse polynomial policy u = Θ(x) @ coeff, clipped to [-1, 1]."""
    Theta_x = library.transform(np.asarray(state, dtype=np.float64).reshape(1, -1))
    u = float((Theta_x @ coeff.T).flatten()[0])
    return np.array([np.clip(u, -1.0, 1.0)], dtype=np.float32)

---

## Part 4 — Evaluate Sparse SINDy Policy in Real Simulator

The SINDy polynomial replaces the NN policy entirely.  The full-order MuJoCo
simulator drives the dynamics; only the action selection is swapped.

In [ ]:
sindy_eval_env = gym.make(ENV_ID)

sindy_ep_rewards = []
sindy_ep_lengths = []

for seed in range(N_SINDY_EVAL):
    obs, _ = sindy_eval_env.reset(seed=seed)
    ep_r = ep_l = 0
    done  = False
    while not done:
        action = sindy_policy(obs_to_state6(obs))
        obs, r, terminated, truncated, _ = sindy_eval_env.step(action)
        ep_r += r
        ep_l += 1
        done  = terminated or truncated
    sindy_ep_rewards.append(ep_r)
    sindy_ep_lengths.append(ep_l)

sindy_eval_env.close()

print("── Track B: SINDy policy evaluation (real simulator) ──────────────")
print(f"  Episodes          : {N_SINDY_EVAL}")
print(f"  Mean length       : {np.mean(sindy_ep_lengths):.1f} / {MAX_STEPS} steps")
print(f"  Mean reward       : {np.mean(sindy_ep_rewards):.2f} ± {np.std(sindy_ep_rewards):.2f}")
print(f"  Perfect (1000 steps): {100*np.mean(np.array(sindy_ep_lengths)==MAX_STEPS):.0f}%")

In [ ]:
# ── Comparison bar chart: NN vs SINDy policy ──────────────────────────────────
best       = os.path.join(CHECKPOINT_DIR, "best_model.zip")
model_path = best if os.path.exists(best) else MODEL_PATH + ".zip"
nn_model   = PPO.load(model_path)

nn_eval_env  = ReducedObsEnv(gym.make(ENV_ID))
nn_ep_rewards, nn_ep_lengths = [], []

for seed in range(N_SINDY_EVAL):
    obs, _ = nn_eval_env.reset(seed=seed)
    ep_r = ep_l = 0
    done  = False
    while not done:
        action, _ = nn_model.predict(obs, deterministic=True)
        obs, r, terminated, truncated, _ = nn_eval_env.step(action)
        ep_r += r
        ep_l += 1
        done  = terminated or truncated
    nn_ep_rewards.append(ep_r)
    nn_ep_lengths.append(ep_l)

nn_eval_env.close()

print("── Track B: NN policy evaluation (real simulator) ─────────────────")
print(f"  Mean length  : {np.mean(nn_ep_lengths):.1f} / {MAX_STEPS} steps")
print(f"  Mean reward  : {np.mean(nn_ep_rewards):.2f} ± {np.std(nn_ep_rewards):.2f}")

# Bar chart
eps = np.arange(1, N_SINDY_EVAL + 1)
fig, (ax_len, ax_rew) = plt.subplots(1, 2, figsize=(13, 5))

ax_len.bar(eps - 0.2, nn_ep_lengths,    0.35, color="steelblue", label="NN policy")
ax_len.bar(eps + 0.2, sindy_ep_lengths, 0.35, color="darkorange", label="SINDy policy")
ax_len.axhline(MAX_STEPS, color="green", ls="--", lw=1.5, label=f"max ({MAX_STEPS})")
ax_len.axhline(np.mean(nn_ep_lengths),    color="steelblue",  ls=":", lw=1.5)
ax_len.axhline(np.mean(sindy_ep_lengths), color="darkorange", ls=":", lw=1.5)
ax_len.set_ylim(0, MAX_STEPS * 1.1)
ax_len.set_xlabel("Episode"); ax_len.set_ylabel("Steps")
ax_len.set_title("Episode length"); ax_len.legend(fontsize=9)

ax_rew.bar(eps - 0.2, nn_ep_rewards,    0.35, color="steelblue", label="NN policy")
ax_rew.bar(eps + 0.2, sindy_ep_rewards, 0.35, color="darkorange", label="SINDy policy")
ax_rew.axhline(np.mean(nn_ep_rewards),    color="steelblue",  ls=":", lw=1.5,
               label=f"NN mean = {np.mean(nn_ep_rewards):.1f}")
ax_rew.axhline(np.mean(sindy_ep_rewards), color="darkorange", ls=":", lw=1.5,
               label=f"SINDy mean = {np.mean(sindy_ep_rewards):.1f}")
ax_rew.set_xlabel("Episode"); ax_rew.set_ylabel("Cumulative reward")
ax_rew.set_title("Episode reward"); ax_rew.legend(fontsize=9)

plt.suptitle(
    f"Track B — NN vs Sparse SINDy policy  |  {N_SINDY_EVAL} evaluation episodes",
    fontsize=11,
)
plt.tight_layout()
plt.show()